# Qiskit circuit optimization via pytket

Build a Qiskit circuit with redundant gates, convert it to pytket,
apply `FullPeepholeOptimise`, and convert back. The gate count
drops significantly while preserving the unitary.

In [ ]:
from pytket import Circuit
from pytket.passes import FullPeepholeOptimise, RemoveRedundancies
from pytket.qiskit import qiskit_to_tk, tk_to_qiskit

import qiskit as qk

## Build a noisy Qiskit circuit

The circuit has intentional redundancies: `X` twice cancels, `H` twice
cancels, `T` then `Tdg` cancels.

In [ ]:
qc = qk.QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.x(0)
qc.x(0)
qc.h(1)
qc.h(1)
qc.cx(2, 0)
qc.t(0)
qc.tdg(0)

print(qc.draw())
print(f"Gates: {qc.size()}  Depth: {qc.depth()}")

## Convert to pytket and optimize

In [ ]:
tk_circ = qiskit_to_tk(qc)
print(f"pytket gates: {tk_circ.n_gates}  depth: {tk_circ.depth()}")

FullPeepholeOptimise().apply(tk_circ)
RemoveRedundancies().apply(tk_circ)

print(f"After optimize: gates={tk_circ.n_gates}  depth={tk_circ.depth()}")

## Convert back to Qiskit

In [ ]:
qc_opt = tk_to_qiskit(tk_circ)
print(qc_opt.draw())
print(f"Gates: {qc_opt.size()}  Depth: {qc_opt.depth()}")
print(f"\nReduction: {qc.size()} -> {qc_opt.size()} gates")